# Week 1: Data Loading, Cleaning and Train/Test Split
DS483 NLP | SMS Spam Detection

## 1. Setup

In [1]:
import pandas as pd

## 2. Raw file check
- one message per line, Tab between label and text, no header
- 5,574 lines expected

In [2]:
with open('../data/SMSSpamCollection', encoding='utf-8') as f:
    lines = f.readlines()

print('Number of lines in the raw file:', len(lines))
print(repr(lines[0]))
print(repr(lines[1]))
print(repr(lines[2]))

Number of lines in the raw file: 5574
'ham\tGo until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n'
'ham\tOk lar... Joking wif u oni...\n'
"spam\tFree entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n"


## 3. Load data

In [3]:
df = pd.read_csv('../data/SMSSpamCollection', sep='\t', header=None, names=['label', 'text'])
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 4. Row count check
- expected 5,574 but got 5,572

In [4]:
len(df)

5572

## 5. Find the missing rows
- look for rows with a line break inside the text

In [5]:
df[df['text'].str.contains('\n', na=False)]

,label,text
5081,ham,"Keep ur problems in ur heart, b'coz nobody wil..."


In [6]:
print(df[df['text'].str.contains('\n', na=False)]['text'].iloc[0])

Keep ur problems in ur heart, b'coz nobody will fight for u. Only u &amp; u have to fight for ur self &amp; win the battle. -VIVEKANAND- G 9t.. SD..
ham	Yeah, give me a call if you've got a minute
ham	HI BABE UAWAKE?FEELLIKW SHIT.JUSTFOUND OUT VIA ALETTER THATMUM GOTMARRIED 4thNOV.BEHIND OURBACKS  FUCKINNICE!SELFISH,DEVIOUSBITCH.ANYWAY,IL CALL U"


- raw lines 5081 to 5083 were joined: line 5081 starts with a quote `"` and line 5083 ends with one

In [7]:
for i in range(5080, 5085):
    print(i, repr(lines[i][:70]), '...', repr(lines[i][-30:]))

5080 'ham\tMark works tomorrow. He gets out at 5. His work is by your house s' ... ' so he can meet u afterwards.\n'
5081 'ham\t"Keep ur problems in ur heart, b\'coz nobody will fight for u. Only' ... 'tle. -VIVEKANAND- G 9t.. SD..\n'
5082 "ham\tYeah, give me a call if you've got a minute\n" ... "a call if you've got a minute\n"
5083 'ham\t"HI BABE UAWAKE?FEELLIKW SHIT.JUSTFOUND OUT VIA ALETTER THATMUM GO' ... 'VIOUSBITCH.ANYWAY,I\x92L CALL U"\n'
5084 'ham\tAmazing : If you rearrange these letters it gives the same meaning' ... ' plus one Its Amazing... !:-)\n'


## 6. Fix
- `quoting=3` tells pandas to ignore quote marks

In [8]:
df = pd.read_csv('../data/SMSSpamCollection', sep='\t', header=None, names=['label', 'text'], quoting=3)

In [9]:
len(df)

5574

In [10]:
df['text'].str.contains('\n', na=False).sum()

np.int64(0)

## 7. Data quality checks
- shape and types, empty values, labels

In [11]:
df.shape

(5574, 2)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   5574 non-null   str  
 1   text    5574 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [13]:
df.isna().sum()

label    0
text     0
dtype: int64

In [14]:
df['label'].unique()

<StringArray>
['ham', 'spam']
Length: 2, dtype: str

## 8. Class distribution
- ham about 87%, spam about 13% (imbalanced)

In [15]:
df['label'].value_counts()

label
ham     4827
spam     747
Name: count, dtype: int64

In [16]:
df['label'].value_counts(normalize=True)

label
ham     0.865985
spam    0.134015
Name: proportion, dtype: float64

## 9. Duplicates
- 403 repeated rows, and no text has two different labels

In [17]:
df.duplicated().sum()

np.int64(403)

In [18]:
df[df.duplicated(keep=False)].sort_values('text').head(10)

,label,text
505,spam,+123 Congratulations - in this week's competit...
2124,spam,+123 Congratulations - in this week's competit...
2344,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1373,ham,1) Go to write msg 2) Put on Dictionary mode 3...
2163,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1050,spam,18 days to Euro2004 kickoff! U will be kept in...
2719,spam,18 days to Euro2004 kickoff! U will be kept in...
2044,spam,4mths half price Orange line rental & latest c...
389,spam,4mths half price Orange line rental & latest c...
1779,ham,7 wonders in My WORLD 7th You 6th Ur style 5th...


In [19]:
df.groupby('text')['label'].nunique().gt(1).sum()

np.int64(0)

## 10. Remove duplicates (before the split)
- 5,574 to 5,171 rows (ham 4,518 / spam 653)

In [20]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print('Before:', len(df), '| After:', len(df_clean))

Before: 5574 | After: 5171


In [21]:
df_clean['label'].value_counts()

label
ham     4518
spam     653
Name: count, dtype: int64

## 11. Train/test split
- 80% train / 20% test
- `stratify` keeps the same ham/spam percentage in both parts
- `random_state=42` gives the same split for everyone
- result: Train 4,136 / Test 1,035

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']
)
print('Train:', len(train_df), '| Test:', len(test_df))

Train: 4136 | Test: 1035


In [24]:
print(train_df['label'].value_counts(normalize=True))
print(test_df['label'].value_counts(normalize=True))
print(test_df['label'].value_counts())

label
ham     0.873791
spam    0.126209
Name: proportion, dtype: float64
label
ham     0.87343
spam    0.12657
Name: proportion, dtype: float64
label
ham     904
spam    131
Name: count, dtype: int64


## 12. Save and check
- all models use these two files
- overlap between train and test must be 0

In [25]:
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)
print('Saved')

Saved


In [26]:
check_train = pd.read_csv('../data/train.csv')
check_test = pd.read_csv('../data/test.csv')
print(len(check_train), len(check_test))
print('Messages in both train and test:', len(set(check_train['text']) & set(check_test['text'])))

4136 1035
Messages in both train and test: 0
